Week 16 · Day 3 — Backend: Retrieval API (query → top-k context → LLM call)
Why this matters

Today you’ll connect the ingestion output to your bot’s brain.
The backend transforms a user query into retrieved chunks, feeds them to the LLM, and returns an answer. This is the core of your RAG logic.

Theory Essentials

Retrieval step: Convert query → embedding → nearest neighbors.

Context window: Concatenate retrieved chunks before generation.

Prompt template: Keep consistent (“Use only this context to answer…”).

LLM call: You can use a local model or Hugging Face API for testing.

Output: Return both answer + source snippets for transparency.

Separation of concerns: Backend only handles logic; frontend handles display.

In [3]:
# Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import joblib
np.random.seed(42)
plt.rcParams["figure.figsize"] = (6,4)
plt.rcParams["axes.grid"] = True

VECTOR_DIR = Path("vector_store")

# ---- Load artifacts ----------------------------------------------------------
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer

def load_vector_store(vector_dir: Path):
    obj = joblib.load(vector_dir / "tfidf_index.joblib")
    vectorizer, matrix, nn = obj["vectorizer"], obj["matrix"], obj["nn"]
    chunks = pd.read_parquet(vector_dir / "chunks.parquet")
    return vectorizer, matrix, nn, chunks

vectorizer, matrix, nn, chunks = load_vector_store(VECTOR_DIR)

# ---- Simple retrieval --------------------------------------------------------
def retrieve(query: str, k: int = 2):
    q_vec = vectorizer.transform([query])
    distances, indices = nn.kneighbors(q_vec, n_neighbors=k)
    results = chunks.iloc[indices[0]].copy()
    results["distance"] = distances[0]
    return results

# Test retrieval
query = "why do we use chunk overlap?"
top_chunks = retrieve(query, k=2)
display(top_chunks[["doc_id", "chunk_id", "distance", "text"]])

# ---- Mock LLM answer ---------------------------------------------------------
def mock_llm_generate(context: str, query: str) -> str:
    # Placeholder for real model (e.g. Hugging Face pipeline)
    return f"Based on the context, chunk overlap ensures continuity between pieces of text, improving retrieval accuracy."

def answer_query(query: str, k: int = 2):
    retrieved = retrieve(query, k)
    context = "\n".join(retrieved["text"].tolist())
    answer = mock_llm_generate(context, query)
    return {"query": query, "answer": answer, "sources": retrieved[["doc_id", "chunk_id"]].to_dict(orient="records")}

response = answer_query("What is chunk overlap?")
print("Q:", response["query"])
print("A:", response["answer"])
print("Sources:", response["sources"])


,doc_id,chunk_id,distance,text
0,faq,0,0.701858,q: what is chunk overlap? a: the number of cha...
1,intro,0,0.934062,# course notes rag combines retrieval with gen...


Q: What is chunk overlap?
A: Based on the context, chunk overlap ensures continuity between pieces of text, improving retrieval accuracy.
Sources: [{'doc_id': 'faq', 'chunk_id': 0}, {'doc_id': 'intro', 'chunk_id': 0}]




## 1. What are **artifacts**?

In this context, *artifacts* = all the saved files produced in Day 2 (ingestion).
They are the reusable pieces your backend can load without recomputing everything:

* **`tfidf_index.joblib`** → contains the fitted TF-IDF vectorizer, the embeddings matrix, and the NearestNeighbors model.
* **`chunks.parquet`** → the metadata table (each row = doc_id, chunk_id, and chunk text).

Think of artifacts as **serialized state** from your ingestion pipeline. Instead of re-cleaning and re-chunking every time, you load these artifacts to do retrieval instantly.

---

## 2. Flow of the Day 3 code

Here’s what happens, top to bottom:

### a) **Load artifacts**

```python
vectorizer, matrix, nn, chunks = load_vector_store(VECTOR_DIR)
```

* Reads the saved TF-IDF model + nearest neighbors index (`joblib.load`).
* Reads the chunks metadata table (`pd.read_parquet`).
* Now your backend has everything it needs: a way to embed queries + search chunks.

---

### b) **Simple retrieval**

```python
def retrieve(query, k=2):
    q_vec = vectorizer.transform([query])          # embed the query in TF-IDF space
    distances, indices = nn.kneighbors(q_vec, k)  # find nearest chunk vectors
    results = chunks.iloc[indices[0]].copy()      # pull chunk text & metadata
    results["distance"] = distances[0]            # add distance score
    return results
```

* Converts user query → vector.
* Finds top-k closest chunks by cosine similarity.
* Returns those chunks with distance scores.

---

### c) **Mock LLM answer**

```python
def mock_llm_generate(context, query):
    return "Based on the context..."
```

* This is a placeholder for an actual LLM call (Hugging Face, OpenAI).
* For now, it just returns a canned answer using the retrieved context.

---

### d) **Answer query end-to-end**

```python
def answer_query(query, k=2):
    retrieved = retrieve(query, k)
    context = "\n".join(retrieved["text"].tolist())  # concat retrieved chunks
    answer = mock_llm_generate(context, query)       # pass to "LLM"
    return {"query": query,
            "answer": answer,
            "sources": retrieved[["doc_id","chunk_id"]].to_dict(orient="records")}
```

* Retrieves top-k chunks.
* Builds a **context window** by concatenating them.
* Sends query + context into LLM (currently mocked).
* Returns a structured response:

  * The query,
  * The generated answer,
  * The **sources** (which chunks the answer came from → transparency).

---

## 3. Big picture (Week 16 · Day 3)

* **Day 2 gave you artifacts** (vector DB + chunks).
* **Day 3 builds retrieval logic**:

  * Embed user query,
  * Retrieve top-k chunks,
  * Feed chunks into LLM,
  * Return both answer + sources.


1) Core (10–15 min)
Task: Change k to 5 and see how the retrieved context changes.

In [5]:
# print(answer_query("Why do we add overlap?", k=5))
print("Number of chunks:", len(chunks))


Number of chunks: 2


AS only 2 chunks were created in day 2 we can't test higehr values of k. n_neighbours <= n_samples.

2) Practice (10–15 min)
Task: Instead of printing all retrieved chunks, show only the top-1 text snippet.

In [6]:
res = retrieve("why chunk overlap?")
print(res.iloc[0]["text"])


q: what is chunk overlap? a: the number of characters repeated between adjacent chunks. q: why citations? a: to show sources and reduce hallucinations.


3) Stretch (optional, 10–15 min)
Task: Replace mock_llm_generate() with a real Hugging Face model (e.g. distilbert-base-uncased or facebook/bart-large-cnn) using the transformers pipeline.

In [7]:
from transformers import pipeline
qa = pipeline("text2text-generation", model="google/flan-t5-small")
def real_llm_generate(context, query):
    prompt = f"Answer the question using only this context:\n{context}\n\nQuestion: {query}"
    return qa(prompt, max_length=150)[0]["generated_text"]



def answer_query(query: str, k: int = 2):
    retrieved = retrieve(query, k)
    context = "\n".join(retrieved["text"].tolist())
    answer = real_llm_generate(context, query)
    return {"query": query, "answer": answer, "sources": retrieved[["doc_id", "chunk_id"]].to_dict(orient="records")}

response = answer_query("What is chunk overlap?")
print("Q:", response["query"])
print("A:", response["answer"])
print("Sources:", response["sources"])


config.json: 0.00B [00:00, ?B/s]

c:\AI-Mastery\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For bet

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Q: What is chunk overlap?
A: a: the number of characters repeated between adjacent chunks
Sources: [{'doc_id': 'faq', 'chunk_id': 0}, {'doc_id': 'intro', 'chunk_id': 0}]


Mini-Challenge (≤40 min)

Goal: Build the backend module app/backend.py exposing an API function.

Acceptance Criteria

✅ Function answer_query(query: str, k: int = 3)
✅ Loads the vector store and retrieves top-k chunks
✅ Calls the (mock or real) LLM
✅ Returns a JSON-like dict:

{
  "query": "What is chunk overlap?",
  "answer": "...",
  "sources": [{"doc_id": "...", "chunk_id": ...}]
}


✅ Can be tested via:

python -m app.backend --query "What is chunk overlap?"

Notes / Key Takeaways

Backend = retrieval + generation glue logic.

Keep your context window concise (<1500 tokens) for smaller models.

Always store source references for transparency.

You can later expose this backend as an API endpoint (FastAPI or Flask).

Tomorrow, you’ll connect it to a Streamlit chat UI.

Reflection

What would happen if the retrieval results are irrelevant?

How could you log queries and results to improve your index later?

1) What would happen if the retrieval results are irrelevant?

The LLM would get the wrong context and might either hallucinate or answer incorrectly.

Even if the LLM sounds fluent, the output could be misleading because it’s grounded on irrelevant snippets.

End users would lose trust if answers don’t match their questions.

2) How could you log queries and results to improve your index later?

Store each user query + retrieved chunks + final answer in a simple log (CSV, database, or JSON).

Add metadata: timestamp, user ID, success rating (if users give feedback).

Later, review logs to:

Spot queries that repeatedly return poor chunks.

Adjust cleaning rules, chunk size, overlap, or even swap embeddings (e.g., TF-IDF → SentenceTransformers).

Use feedback to build a better evaluation set for tuning your pipeline.